<a href="https://colab.research.google.com/github/janpfrang-hash/improved-data-evaluation-fatigue-tester/blob/main/Kopie_von_Improved_Data_evaluation_fatigue_tester.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# @title
"""
Wöhler-Kurven Datenauswertung für Google Colab - Multi-File Version
DRUCK-OPTIMIERTE VERSION für schnelles Drucken aus Acrobat
Version of 24 feb 2026 - MIT RASTERISIERUNG, DRIFT-ANALYSE, MASCHINENPARAMETER & DIN 50100 CHECK
- Plots als Rastergrafiken (nicht Vektoren)
- Keine Transparenzen (schnelleres Rendering)
- Optimiert für Windows PDF-Drucker
- Drift-Analyse mit linearer Regression
- Maschinenparameter aus CNF-Header dekodieren & anzeigen
- NEU: DIN 50100 Konformitätsprüfung (3% Kriterium für 3-Sigma und 10-Zyklen Ø)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.gridspec import GridSpec
from io import StringIO
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
from datetime import datetime
from scipy import stats
from dataclasses import dataclass, field
from typing import Tuple, Optional, Dict

# ============================================================================
# DRUCK-OPTIMIERUNGS-EINSTELLUNGEN
# ============================================================================
# Bildschirm-DPI (niedrig für schnelle Anzeige)
SCREEN_DPI = 72

# DRUCK-DPI für rasterisierte Plots (höher = bessere Qualität beim Drucken)
RASTER_DPI = 150  # 150 = gute Balance, 200 = höhere Qualität, 100 = schneller

# Rasterisierung aktivieren (KRITISCH für schnelles Drucken!)
RASTERIZE_PLOTS = True  # True = schnelles Drucken, False = Vektor (langsam)

# Transparenzen (alpha) - beim Drucken problematisch
USE_TRANSPARENCY = False  # False = schnelleres Drucken

# Downsampling (wie vorher)
MAX_PLOT_POINTS = 5000

# ============================================================================
# Globale Variablen
# ============================================================================
data_dict = {}
file_checkboxes = {}
cnf_dict = {}  # NEU: CNF-Konfigurationen pro Datei

COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
          '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']


# ============================================================================
# CNF Configuration Dataclass
# ============================================================================
@dataclass
class CNFConfig:
    """
    Machine configuration decoded from a CNF header line.
    """
    # Core force targets
    kraft2_N: float         = 0.0   # Kraft @2  — upper force target [N]
    kraftsoll2_N: float     = 0.0   # Kraftsoll @2 — duplicate of kraft2_N [N]
    kraft1_N: float         = 0.0   # Kraft @1  — lower force target (derived) [N]
    kraft1_ratio: int       = 0     # Divisor: Kraft@1 = Kraft@2 / ratio

    # Safety & tolerances
    kraftabschaltung_N: float = 0.0 # Emergency force cutoff [N]
    toleranz_pct: int         = 0   # Force tolerance band ±% [%]

    # Position-correction budgets (Nachregelweg)
    nrw_k1_mm: float        = 0.0   # Nachregelweg Kraft 1 [mm]
    nrw_k2_mm: float        = 0.0   # Nachregelweg Kraft 2 [mm]

    # Cycle control
    betaetigungen: int      = 0     # Planned total cycles [#]
    messen_alle_n: int      = 0     # Log every N cycles [#]

    # Device identification (parsed from surrounding header lines)
    device_version: str     = ""    # e.g. "V1.5"
    device_id: str          = ""    # e.g. "2BC4D630"
    display_id: str         = ""    # e.g. "ceSID-48339C75"

    # Internal
    config_id: int          = 0     # CNF field 1
    raw_cnf: str            = ""    # full raw CNF line for audit trail
    parse_ok: bool          = False # True if successfully decoded

    def force_tolerance_window(self) -> Tuple[float, float]:
        """Return (lower_limit, upper_limit) for Kraft @2 based on Toleranz %."""
        delta = self.kraft2_N * self.toleranz_pct / 100.0
        return (self.kraft2_N - delta, self.kraft2_N + delta)

    def to_display_rows(self) -> list:
        """Return a list of (label, value_str, unit) tuples for display."""
        lo, hi = self.force_tolerance_window()
        return [
            ("Kraft @2  (upper)",        f"{self.kraft2_N:.1f}",         "N"),
            ("Kraftsoll @2",             f"{self.kraftsoll2_N:.1f}",     "N"),
            ("Kraft @1  (lower)",        f"{self.kraft1_N:.1f}",         "N  (= @2 ÷ ratio)"),
            ("Force ratio",              f"1 : {self.kraft1_ratio}",     ""),
            ("Kraftabschaltung",         f"{self.kraftabschaltung_N:.1f}","N  (emergency cutoff)"),
            ("Kraftsoll @2 Toleranz",    f"± {self.toleranz_pct}",       "%"),
            ("  Force window",           f"{lo:.1f} – {hi:.1f}",        "N"),
            ("Nachregelweg Kraft 1",     f"{self.nrw_k1_mm:.2f}",        "mm"),
            ("Nachregelweg Kraft 2",     f"{self.nrw_k2_mm:.2f}",        "mm"),
            ("Betätigungen",             f"{self.betaetigungen:,}",      "# (planned cycles)"),
            ("Messen alle N",            f"{self.messen_alle_n}",        "# (log interval)"),
            ("Device",                   f"{self.device_version}",       f"ID: {self.device_id}"),
            ("Display",                  f"{self.display_id}",           ""),
        ]


# ============================================================================
# CNF Parsing Functions
# ============================================================================
def parse_cnf_line(raw: str) -> CNFConfig:
    cfg = CNFConfig(raw_cnf=raw)
    try:
        values = [int(x) for x in raw.replace('!', '').split(';')[1:]
                  if x.strip().lstrip('-').isdigit()]

        if len(values) < 32:
            return cfg   # parse_ok stays False

        def u16(i: int) -> int:
            return values[i] + (values[i + 1] << 8)

        def u32(i: int) -> int:
            return (values[i]
                    + (values[i + 1] << 8)
                    + (values[i + 2] << 16)
                    + (values[i + 3] << 24))

        cfg.config_id           = values[0]            # field 1
        cfg.kraft2_N            = u16(7)  / 10.0       # fields 8-9
        cfg.kraftsoll2_N        = u16(9)  / 10.0       # fields 10-11
        cfg.nrw_k1_mm           = u16(11) / 100.0      # fields 12-13
        cfg.nrw_k2_mm           = u16(13) / 100.0      # fields 14-15
        cfg.kraftabschaltung_N  = u16(15) / 10.0       # fields 16-17
        cfg.toleranz_pct        = values[17]           # field  18
        cfg.messen_alle_n       = u16(21)              # fields 22-23
        cfg.kraft1_ratio        = values[23]           # field  24
        cfg.betaetigungen       = u32(28)              # fields 29-32

        if cfg.kraft1_ratio > 0:
            cfg.kraft1_N = cfg.kraft2_N / cfg.kraft1_ratio

        cfg.parse_ok = True

    except Exception as e:
        print(f"CNFConfig parse error: {e}  —  raw: {raw[:80]}")

    return cfg


def parse_header_block(lines: list) -> CNFConfig:
    cfg = CNFConfig()

    for line in lines:
        line = line.strip()

        if '### VoiceCoilTestStand' in line:
            for token in line.split():
                if token.startswith('V') and len(token) > 1:
                    cfg.device_version = token
                if token.startswith('ID='):
                    cfg.device_id = token.split('=', 1)[1].rstrip('#').strip()

        elif line.startswith('For display'):
            parts = line.split()
            if len(parts) >= 3:
                cfg.display_id = parts[2]

        elif line.startswith('CNF;'):
            parsed = parse_cnf_line(line)
            if parsed.parse_ok:
                parsed.device_version = cfg.device_version
                parsed.device_id      = cfg.device_id
                parsed.display_id     = cfg.display_id
                cfg = parsed

    return cfg


# ============================================================================
# Downsampling
# ============================================================================
def downsample_data(df, max_points=MAX_PLOT_POINTS):
    """Reduziert Datenpunkte intelligent für schnelleres Plotten"""
    if len(df) <= max_points:
        return df

    factor = len(df) // max_points + 1
    df_downsampled = df.iloc[::factor].copy()

    if df.index[0] not in df_downsampled.index:
        df_downsampled = pd.concat([df.iloc[[0]], df_downsampled])
    if df.index[-1] not in df_downsampled.index:
        df_downsampled = pd.concat([df_downsampled, df.iloc[[-1]]])

    return df_downsampled.sort_index()


# ============================================================================
# Log-File parsen
# ============================================================================
def parse_logfile(file_content):
    """Parst den Inhalt eines Log-Files inkl. CNF-Header"""
    lines = file_content.strip().split('\n')

    header_lines = []
    for line in lines:
        if line.startswith('DTA;'):
            break
        header_lines.append(line)

    cnf_config = parse_header_block(header_lines)

    dta_lines = [line for line in lines if line.startswith('DTA;')]
    data = []

    for line in dta_lines:
        parts = line.replace('DTA;', '').replace('!', '').split(';')
        if len(parts) >= 9:
            try:
                row = {
                    'Zyklen': int(parts[0]),
                    'Position_1_mm': int(parts[1]) / 100.0,
                    'Kraft_1_N': int(parts[2]) / 10.0,
                    'Zusatzweg_1_mm': int(parts[3]) / 100.0,
                    'Position_2_mm': int(parts[4]) / 100.0,
                    'Kraft_2_N': int(parts[5]) / 10.0,
                    'Zusatzweg_2_mm': int(parts[6]) / 100.0,
                    'Weg_mm': int(parts[7]) / 100.0,
                    'Fehlercode': int(parts[8])
                }
                data.append(row)
            except (ValueError, IndexError):
                continue

    df = pd.DataFrame(data)

    if len(df) > 0:
        df['R_Wert'] = np.where(
            df['Kraft_2_N'] != 0,
            df['Kraft_1_N'] / df['Kraft_2_N'],
            np.nan
        )
        df['Kraftamplitude_N'] = (df['Kraft_2_N'] - df['Kraft_1_N']) / 2
        df['Mittelkraft_N'] = (df['Kraft_2_N'] + df['Kraft_1_N']) / 2
        df['Krafthub_N'] = df['Kraft_2_N'] - df['Kraft_1_N']

        s_min = df['Position_1_mm'] + df['Zusatzweg_1_mm']
        s_max = df['Position_2_mm'] + df['Zusatzweg_2_mm']
        df['Weghub_mm'] = s_max - s_min

        df['Steifigkeit_N_per_mm'] = np.where(
            df['Weghub_mm'] != 0,
            df['Krafthub_N'] / df['Weghub_mm'],
            np.nan
        )

        window_size = 100
        df['Steifigkeit_geglättet'] = df['Steifigkeit_N_per_mm'].rolling(
            window=window_size, center=True, min_periods=1
        ).mean()

        initial_cycles = min(50, len(df))
        k_initial = df['Steifigkeit_geglättet'].head(initial_cycles).mean()

        df['Steifigkeit_normalisiert'] = df['Steifigkeit_geglättet'] / k_initial
        df['Steifigkeitsverlust_Prozent'] = (1 - df['Steifigkeit_normalisiert']) * 100

        df.attrs['k_initial'] = k_initial

    return df, cnf_config


# ============================================================================
# Daten glätten
# ============================================================================
def smooth_data(df, window_size=50):
    """Glättet die Daten"""
    df_smooth = df.copy()
    columns_to_smooth = [
        'Position_1_mm', 'Kraft_1_N', 'Zusatzweg_1_mm',
        'Position_2_mm', 'Kraft_2_N', 'Zusatzweg_2_mm',
        'Weg_mm', 'R_Wert', 'Kraftamplitude_N', 'Mittelkraft_N',
        'Steifigkeitsverlust_Prozent'
    ]

    for col in columns_to_smooth:
        if col in df_smooth.columns:
            df_smooth[col] = df_smooth[col].rolling(
                window=window_size, center=True, min_periods=1
            ).mean()

    return df_smooth


# ============================================================================
# Statistik & DIN 50100 Checks
# ============================================================================
def calculate_statistics(df, column_name):
    """Berechnet statistische Kennwerte"""
    data = df[column_name].dropna()
    if len(data) == 0:
        return None

    min_val = data.min()
    max_val = data.max()
    mean_val = data.mean()
    std_val = data.std()
    var_val = data.var()

    upper_3sigma = mean_val + 3 * std_val
    lower_3sigma = mean_val - 3 * std_val

    if len(data) > 1:
        value_diffs = np.abs(data.diff()).dropna()
        value_diffs = value_diffs[value_diffs > 0]
        resolution = value_diffs.min() if len(value_diffs) > 0 else 0.0
    else:
        resolution = 0.0

    if len(data) <= 5000:
        try:
            _, p_value = stats.shapiro(data)
            distribution = "Normal" if p_value > 0.05 else "Nicht-normal"
        except (ValueError, RuntimeError):
            distribution = "Unbekannt"
    else:
        try:
            result = stats.anderson(data, dist='norm')
            distribution = "Normal" if result.statistic < result.critical_values[2] else "Nicht-normal"
        except (ValueError, RuntimeError):
            distribution = "Unbekannt"

    return {
        'min': min_val, 'max': max_val, 'mean': mean_val,
        'std': std_val, 'var': var_val,
        'upper_3sigma': upper_3sigma, 'lower_3sigma': lower_3sigma,
        'distribution': distribution, 'resolution': resolution,
        'n': len(data)
    }

def evaluate_din50100_criteria(df, cnf):
    """
    Prüft, ob die F_max (Kraft_2_N) Messwerte innerhalb der ±3% Toleranz
    des Sollwerts liegen, wie in DIN 50100 gefordert.
    """
    if not cnf or not cnf.parse_ok or cnf.kraft2_N == 0:
        return None

    target_fmax = cnf.kraft2_N
    allowed_dev = target_fmax * 0.03  # ±3% limit

    # 1. Kriterium: 3-Sigma-Abweichung <= 3% des Sollwerts
    fmax_std = df['Kraft_2_N'].std()
    sigma3_val = 3 * fmax_std
    crit1_pass = sigma3_val <= allowed_dev

    # 2. Kriterium: 10-Zyklen gleitender Mittelwert <= 3% Abweichung
    rolling_mean_10 = df['Kraft_2_N'].rolling(window=10, min_periods=1).mean()
    max_rolling_dev = (rolling_mean_10 - target_fmax).abs().max()
    crit2_pass = max_rolling_dev <= allowed_dev

    return {
        'target': target_fmax,
        'allowed_dev': allowed_dev,
        '3sigma': sigma3_val,
        'crit1_pass': crit1_pass,
        'max_roll_dev': max_rolling_dev,
        'crit2_pass': crit2_pass
    }


# ============================================================================
# Drift-Analyse
# ============================================================================
def calculate_drift(df, column_name):
    """Berechnet Drift-Parameter für eine Spalte über die Zyklen"""
    data = df[[column_name, 'Zyklen']].dropna()

    if len(data) < 10:
        return None

    cycles = data['Zyklen'].values
    values = data[column_name].values

    n_avg = min(50, len(data) // 10)
    start_value = values[:n_avg].mean()
    end_value = values[-n_avg:].mean()
    start_cycle = cycles[:n_avg].mean()
    end_cycle = cycles[-n_avg:].mean()

    try:
        slope, intercept, r_value, p_value, std_err = stats.linregress(cycles, values)
        r_squared = r_value ** 2
    except (ValueError, RuntimeError):
        slope = 0
        intercept = start_value
        r_squared = 0
        p_value = 1
        std_err = 0

    total_drift = end_value - start_value
    total_cycles = end_cycle - start_cycle

    if start_value != 0:
        relative_drift_percent = (total_drift / abs(start_value)) * 100
    else:
        relative_drift_percent = 0

    if total_cycles > 0:
        drift_per_1000 = (total_drift / total_cycles) * 1000
        relative_drift_per_1000 = (relative_drift_percent / total_cycles) * 1000
    else:
        drift_per_1000 = 0
        relative_drift_per_1000 = 0

    if abs(slope) < 1e-6:
        trend = "Stabil"
    elif slope > 0:
        trend = "Steigend ↗"
    else:
        trend = "Fallend ↘"

    if r_squared > 0.8 and p_value < 0.001:
        significance = "Stark"
    elif r_squared > 0.5 and p_value < 0.05:
        significance = "Mittel"
    elif r_squared > 0.3:
        significance = "Schwach"
    else:
        significance = "Nicht signifikant"

    return {
        'start_value': start_value,
        'end_value': end_value,
        'total_drift': total_drift,
        'relative_drift_percent': relative_drift_percent,
        'drift_per_1000': drift_per_1000,
        'relative_drift_per_1000': relative_drift_per_1000,
        'slope': slope,
        'intercept': intercept,
        'r_squared': r_squared,
        'p_value': p_value,
        'trend': trend,
        'significance': significance,
        'total_cycles': total_cycles
    }


# ============================================================================
# Drift-Visualisierung
# ============================================================================
def create_drift_plot(ax, df, column_name, column_label, color='blue'):
    """Erstellt Drift-Plot mit Trend-Linie"""
    drift = calculate_drift(df, column_name)

    if drift is None:
        ax.text(0.5, 0.5, 'Nicht genügend Daten',
                ha='center', va='center', transform=ax.transAxes)
        return None

    df_plot = downsample_data(df, MAX_PLOT_POINTS)
    alpha_val = 0.7 if USE_TRANSPARENCY else 1.0

    ax.plot(df_plot['Zyklen'], df_plot[column_name],
            color=color, alpha=alpha_val, linewidth=0.8,
            label='Messdaten', rasterized=RASTERIZE_PLOTS)

    cycles = df['Zyklen'].values
    trend_line = drift['slope'] * cycles + drift['intercept']
    ax.plot(cycles[::len(cycles)//100], trend_line[::len(cycles)//100],
            'r--', linewidth=2, label=f"Trend (R²={drift['r_squared']:.3f})",
            rasterized=RASTERIZE_PLOTS)

    ax.plot(df['Zyklen'].iloc[0], drift['start_value'],
            'go', markersize=8, label=f"Start: {drift['start_value']:.2f}")
    ax.plot(df['Zyklen'].iloc[-1], drift['end_value'],
            'ro', markersize=8, label=f"Ende: {drift['end_value']:.2f}")

    ax.set_xlabel('Zyklen', fontsize=11)
    ax.set_ylabel(column_label, fontsize=11)
    ax.set_title(f'Drift-Analyse: {column_label}', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, linewidth=0.5)
    ax.legend(loc='best', fontsize=8)

    info_text = (
        f"Drift: {drift['total_drift']:+.3f} ({drift['relative_drift_percent']:+.2f}%)\n"
        f"Rate: {drift['drift_per_1000']:+.4f} / 1000 Zyklen\n"
        f"Trend: {drift['trend']} ({drift['significance']})"
    )
    ax.text(0.02, 0.98, info_text, transform=ax.transAxes,
            fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    return drift


def create_drift_table(selected_files, data_dict, show_drift_kraft1,
                       show_drift_kraft2, show_drift_r_wert,
                       show_drift_kraftamplitude, use_smoothing=False):
    """Erstellt Zusammenfassungs-Tabelle für Drift-Analyse"""

    if not selected_files:
        print("❌ Keine Dateien ausgewählt.")
        return

    columns_to_analyze = []
    if show_drift_kraft1:
        columns_to_analyze.append(('Kraft_1_N', 'Kraft 1 [N]'))
    if show_drift_kraft2:
        columns_to_analyze.append(('Kraft_2_N', 'Kraft 2 [N]'))
    if show_drift_r_wert:
        columns_to_analyze.append(('R_Wert', 'R-Wert'))
    if show_drift_kraftamplitude:
        columns_to_analyze.append(('Kraftamplitude_N', 'F_a [N]'))

    if not columns_to_analyze:
        print("❌ Keine Drift-Parameter ausgewählt.")
        return

    print(f"\n📊 DRIFT-ANALYSE")
    print("=" * 80)

    for col_name, col_label in columns_to_analyze:
        print(f"\n{col_label}:")
        print("-" * 80)

        table_data = []
        headers = ['Datei', 'Start', 'Ende', 'Drift abs.', 'Drift %',
                   'pro 1000 Zyklen', 'Trend', 'R²', 'Signifikanz']

        for filename in selected_files:
            df = data_dict[filename]
            if use_smoothing:
                df = smooth_data(df, window_size=50)

            drift = calculate_drift(df, col_name)

            if drift:
                short_name = filename if len(filename) <= 25 else filename[:22] + "..."
                row = [
                    short_name,
                    f"{drift['start_value']:.2f}",
                    f"{drift['end_value']:.2f}",
                    f"{drift['total_drift']:+.3f}",
                    f"{drift['relative_drift_percent']:+.2f}%",
                    f"{drift['drift_per_1000']:+.4f}",
                    drift['trend'],
                    f"{drift['r_squared']:.3f}",
                    drift['significance']
                ]
                table_data.append(row)

        if table_data:
            fig_height = 2 + 0.35 * len(table_data)
            fig = plt.figure(figsize=(16, fig_height))
            ax = fig.add_subplot(111)
            ax.axis('off')

            table = ax.table(cellText=table_data, colLabels=headers,
                           cellLoc='center', loc='center',
                           bbox=[0, 0, 1, 1])

            table.auto_set_font_size(False)
            table.set_fontsize(7)
            table.scale(1, 1.2)

            for i in range(len(headers)):
                table[(0, i)].set_facecolor('#FF6B6B')
                table[(0, i)].set_text_props(weight='bold', color='white')

            for i in range(1, len(table_data) + 1):
                for j in range(len(headers)):
                    if i % 2 == 0:
                        table[(i, j)].set_facecolor('#FFE5E5')
                    else:
                        table[(i, j)].set_facecolor('#FFFFFF')

            smooth_text = ' (geglättet)' if use_smoothing else ''
            fig.suptitle(f'Drift-Analyse: {col_label}{smooth_text}',
                         fontsize=14, fontweight='bold', y=0.98)

            plt.tight_layout()
            plt.show()


def create_all_drift_plots(show_drift_kraft1, show_drift_kraft2,
                           show_drift_r_wert, show_drift_kraftamplitude,
                           use_smoothing=False):
    """Erstellt Drift-Plots für alle ausgewählten Parameter"""
    selected_files = get_selected_files()
    if not selected_files:
        print("❌ Keine Dateien ausgewählt.")
        return

    params = []
    if show_drift_kraft1:
        params.append(('Kraft_1_N', 'Kraft 1 [N]'))
    if show_drift_kraft2:
        params.append(('Kraft_2_N', 'Kraft 2 [N]'))
    if show_drift_r_wert:
        params.append(('R_Wert', 'R-Wert'))
    if show_drift_kraftamplitude:
        params.append(('Kraftamplitude_N', 'Kraftamplitude [N]'))

    if not params:
        print("❌ Bitte wähle mindestens einen Drift-Parameter aus.")
        return

    n_params = len(params)
    n_files = len(selected_files)

    print(f"📊 Erstelle Drift-Analyse: {n_params} Parameter × {n_files} Datei(en)")

    for col_name, col_label in params:
        fig, axes = plt.subplots(n_files, 1, figsize=(14, 6 * n_files), dpi=SCREEN_DPI)
        if n_files == 1:
            axes = [axes]

        for idx, filename in enumerate(selected_files):
            df = data_dict[filename]
            if use_smoothing:
                df = smooth_data(df, window_size=50)

            color = COLORS[idx % len(COLORS)]
            drift = create_drift_plot(axes[idx], df, col_name,
                                     f"{col_label} - {filename}", color)

        smooth_text = ' (geglättet)' if use_smoothing else ''
        fig.suptitle(f'Drift-Analyse: {col_label}{smooth_text}',
                    fontsize=16, fontweight='bold', y=0.995)

        plt.tight_layout()
        plt.show()

    create_drift_table(selected_files, data_dict, show_drift_kraft1,
                      show_drift_kraft2, show_drift_r_wert,
                      show_drift_kraftamplitude, use_smoothing)


# ============================================================================
# Q-Q Plot
# ============================================================================
def create_qq_plot_with_histogram(ax_main, ax_hist, data, title, color='#1f77b4'):
    """DRUCK-OPTIMIERT: Q-Q Plot mit Histogram"""
    data_clean = data.dropna()

    if len(data_clean) < 3:
        ax_main.text(0.5, 0.5, 'Nicht genügend Daten',
                    ha='center', va='center', transform=ax_main.transAxes)
        return

    if len(data_clean) > MAX_PLOT_POINTS:
        sample_indices = np.linspace(0, len(data_clean)-1, MAX_PLOT_POINTS, dtype=int)
        data_sampled = data_clean.iloc[sample_indices]
    else:
        data_sampled = data_clean

    alpha_val = 0.7 if USE_TRANSPARENCY else 1.0

    stats.probplot(data_sampled, dist="norm", plot=ax_main)
    ax_main.get_lines()[0].set_markerfacecolor(color)
    ax_main.get_lines()[0].set_markeredgecolor(color)
    ax_main.get_lines()[0].set_markersize(2)
    ax_main.get_lines()[0].set_alpha(alpha_val)
    ax_main.get_lines()[0].set_rasterized(RASTERIZE_PLOTS)
    ax_main.get_lines()[1].set_color('red')
    ax_main.get_lines()[1].set_linewidth(1.5)
    ax_main.get_lines()[1].set_rasterized(RASTERIZE_PLOTS)

    ax_main.set_title(title, fontsize=11, fontweight='bold')
    ax_main.set_xlabel('Theoretische Quantile', fontsize=9)
    ax_main.set_ylabel('Beobachtete Quantile', fontsize=9)
    ax_main.grid(True, alpha=0.3, linewidth=0.5)

    if len(data_clean) <= 5000:
        try:
            _, p_value = stats.shapiro(data_clean)
            text = f'Shapiro-Wilk\np={p_value:.4f}\nNormal: {"Ja" if p_value > 0.05 else "Nein"}'
        except (ValueError, RuntimeError):
            text = 'Test fehlgeschlagen'
    else:
        text = 'Anderson-Darling\n(siehe Tabelle)'

    ax_main.text(0.02, 0.98, text, transform=ax_main.transAxes,
                fontsize=8, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    n_bins = min(20, max(10, len(data_clean) // 100))
    n, bins, patches = ax_hist.hist(data_clean, bins=n_bins, color=color,
                                     alpha=alpha_val, edgecolor='black', linewidth=0.5)

    for patch in patches:
        patch.set_rasterized(RASTERIZE_PLOTS)

    ax_hist.set_xlabel('Wert', fontsize=8)
    ax_hist.set_ylabel('Häufigkeit', fontsize=8)
    ax_hist.set_title('Verteilung', fontsize=9, fontweight='bold')
    ax_hist.grid(True, alpha=0.3, axis='y', linewidth=0.5)

    mu, sigma = data_clean.mean(), data_clean.std()
    x = np.linspace(data_clean.min(), data_clean.max(), 50)
    scale_factor = len(data_clean) * (data_clean.max() - data_clean.min()) / n_bins
    p = stats.norm.pdf(x, mu, sigma) * scale_factor
    line = ax_hist.plot(x, p, 'r-', linewidth=1.5, label='Normalverteilung')
    line[0].set_rasterized(RASTERIZE_PLOTS)
    ax_hist.legend(fontsize=7)


def create_qq_plots_for_parameter(selected_files, column_name, column_label,
                                  data_dict, use_smoothing=False):
    if not selected_files:
        print("❌ Keine Dateien ausgewählt.")
        return

    n_files = len(selected_files)
    fig = plt.figure(figsize=(14, 5 * n_files), dpi=SCREEN_DPI)

    for idx, filename in enumerate(selected_files):
        df = data_dict[filename]
        if use_smoothing:
            df = smooth_data(df, window_size=50)

        gs = GridSpec(n_files, 2, figure=fig,
                     width_ratios=[2, 1],
                     hspace=0.4, wspace=0.3,
                     left=0.1, right=0.95, top=0.95, bottom=0.05)

        ax_qq = fig.add_subplot(gs[idx, 0])
        ax_hist = fig.add_subplot(gs[idx, 1])

        color = COLORS[idx % len(COLORS)]
        smooth_text = ' (geglättet)' if use_smoothing else ''
        title = f'Q-Q: {column_label}{smooth_text} - {filename}'

        create_qq_plot_with_histogram(ax_qq, ax_hist, df[column_name], title, color)

    plt.show()


# ============================================================================
# Tabellen für Plots
# ============================================================================
def create_statistics_table(ax, selected_files, column_name, data_dict, use_smoothing=False):
    stats_list = []
    for filename in selected_files:
        df = data_dict[filename]
        if use_smoothing:
            df = smooth_data(df, window_size=50)
        s = calculate_statistics(df, column_name)
        if s:
            stats_list.append((filename, s))

    if not stats_list:
        return

    table_data = []
    headers = ['Datei', 'Min', 'Max', 'Mittelwert', 'Std.Abw.', 'Varianz',
               '-3σ', '+3σ', 'Verteilung', 'Auflösung', 'n']

    for filename, s in stats_list:
        short_name = filename if len(filename) <= 25 else filename[:22] + "..."

        if column_name == 'R_Wert' or abs(s['max']) < 10:
            decimal_places = 3
        else:
            decimal_places = 2

        row = [
            short_name,
            f"{s['min']:.{decimal_places}f}",
            f"{s['max']:.{decimal_places}f}",
            f"{s['mean']:.{decimal_places}f}",
            f"{s['std']:.{decimal_places}f}",
            f"{s['var']:.{decimal_places}f}",
            f"{s['lower_3sigma']:.{decimal_places}f}",
            f"{s['upper_3sigma']:.{decimal_places}f}",
            s['distribution'],
            f"{s['resolution']:.4f}",
            f"{s['n']}"
        ]
        table_data.append(row)

    table = ax.table(cellText=table_data, colLabels=headers,
                     cellLoc='center', loc='bottom',
                     bbox=[0, -0.55, 1, 0.3])

    table.auto_set_font_size(False)
    table.set_fontsize(7)
    table.scale(1, 1.2)

    for i in range(len(headers)):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(weight='bold', color='white')

    for i in range(1, len(table_data) + 1):
        for j in range(len(headers)):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#E7E6E6')
            else:
                table[(i, j)].set_facecolor('#FFFFFF')

    pos = ax.get_position()
    ax.set_position([pos.x0, pos.y0 + 0.15, pos.width, pos.height * 0.75])


# ============================================================================
# Zusammenfassung (Screen)
# ============================================================================
def create_summary_table(selected_files, data_dict, show_kriterium_10,
                         show_kriterium_20, show_kriterium_50, use_smoothing=False):
    if not selected_files:
        return

    fig_height = 2 + 0.4 * len(selected_files)
    fig = plt.figure(figsize=(16, fig_height))
    ax = fig.add_subplot(111)
    ax.axis('off')

    table_data = []
    headers = ['Datei', 'Kraft 1 Ø [N]', 'Kraft 1 σ [N]',
               'Kraft 2 Ø [N]', 'Kraft 2 σ [N]',
               'DIN 3σ Check', 'DIN 10-Zykl. Check',
               'F_a Ø [N]', 'F_a σ [N]']

    if show_kriterium_10: headers.append('Zyklen @ 10%')
    if show_kriterium_20: headers.append('Zyklen @ 20%')
    if show_kriterium_50: headers.append('Zyklen @ 50%')

    for filename in selected_files:
        df = data_dict[filename]
        if use_smoothing:
            df = smooth_data(df, window_size=50)

        short_name = filename if len(filename) <= 25 else filename[:22] + "..."

        # Werte für DIN Prüfung holen
        din_3sigma_str = "---"
        din_10cyc_str = "---"
        if filename in cnf_dict and cnf_dict[filename].parse_ok:
            din_eval = evaluate_din50100_criteria(df, cnf_dict[filename])
            if din_eval:
                din_3sigma_str = "PASS" if din_eval['crit1_pass'] else "FAIL"
                din_10cyc_str = "PASS" if din_eval['crit2_pass'] else "FAIL"

        row = [short_name,
               f"{df['Kraft_1_N'].mean():.2f}", f"{df['Kraft_1_N'].std():.2f}",
               f"{df['Kraft_2_N'].mean():.2f}", f"{df['Kraft_2_N'].std():.2f}",
               din_3sigma_str, din_10cyc_str,
               f"{df['Kraftamplitude_N'].mean():.2f}", f"{df['Kraftamplitude_N'].std():.2f}"]

        for thresh, show in [(10, show_kriterium_10), (20, show_kriterium_20), (50, show_kriterium_50)]:
            if show:
                idx = df[df['Steifigkeitsverlust_Prozent'] >= thresh].first_valid_index()
                row.append(f"{int(df.loc[idx, 'Zyklen']):,}" if idx is not None else "---")

        table_data.append(row)

    table = ax.table(cellText=table_data, colLabels=headers,
                     cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
    table.auto_set_font_size(False)
    table.set_fontsize(7)
    table.scale(1, 1.2)

    for i in range(len(headers)):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(weight='bold', color='white')

    for i in range(1, len(table_data) + 1):
        for j in range(len(headers)):
            table[(i, j)].set_facecolor('#E7E6E6' if i % 2 == 0 else '#FFFFFF')

            # Zellen rot/grün färben für PASS/FAIL
            cell_text = table_data[i-1][j]
            if cell_text == "PASS":
                table[(i, j)].set_text_props(color='green', weight='bold')
            elif cell_text == "FAIL":
                table[(i, j)].set_text_props(color='red', weight='bold')

    title = f'Zusammenfassung{" (geglättet)" if use_smoothing else ""}'
    fig.suptitle(title, fontsize=13, fontweight='bold', y=0.98)

    plt.tight_layout()
    plt.show()


# ============================================================================
# Maschinenparameter anzeigen
# ============================================================================
def show_machine_parameters():
    selected_files = get_selected_files()
    if not selected_files:
        print("❌ Keine Dateien ausgewählt.")
        return

    files_with_cnf = [f for f in selected_files if f in cnf_dict and cnf_dict[f].parse_ok]
    files_without_cnf = [f for f in selected_files if f not in files_with_cnf]

    if files_without_cnf:
        print(f"⚠️  Kein CNF-Header gefunden in: {', '.join(files_without_cnf)}")

    if not files_with_cnf:
        print("❌ Keine Maschinenparameter verfügbar.")
        print("   Die Log-Dateien enthalten keinen CNF-Header.")
        return

    print(f"🔧 Maschinenparameter für {len(files_with_cnf)} Datei(en)\n")

    for filename in files_with_cnf:
        cfg = cnf_dict[filename]
        rows = cfg.to_display_rows()

        print(f"\n{'='*70}")
        print(f"📋 {filename}")
        print(f"{'='*70}")
        for label, value, unit in rows:
            print(f"  {label:<28} {value:>15}  {unit}")

        fig_height = 1.5 + 0.35 * len(rows)
        fig = plt.figure(figsize=(12, fig_height))
        ax = fig.add_subplot(111)
        ax.axis('off')

        table_data = [[label, value, unit] for label, value, unit in rows]
        col_labels = ['Parameter', 'Wert', 'Einheit / Bemerkung']

        table = ax.table(cellText=table_data, colLabels=col_labels,
                        cellLoc='left', loc='center',
                        colWidths=[0.35, 0.25, 0.40],
                        bbox=[0, 0, 1, 1])

        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1, 1.4)

        for j in range(3):
            table[(0, j)].set_facecolor('#2E86AB')
            table[(0, j)].set_text_props(weight='bold', color='white', fontsize=10)

        category_rows = {0, 1, 2, 3}
        safety_rows = {4, 5, 6}
        nachregelweg_rows = {7, 8}
        cycle_rows = {9, 10}
        device_rows = {11, 12}

        for i in range(1, len(table_data) + 1):
            row_idx = i - 1
            for j in range(3):
                if row_idx in category_rows:
                    table[(i, j)].set_facecolor('#E3F2FD')
                elif row_idx in safety_rows:
                    table[(i, j)].set_facecolor('#FFF9C4')
                elif row_idx in nachregelweg_rows:
                    table[(i, j)].set_facecolor('#E8F5E9')
                elif row_idx in cycle_rows:
                    table[(i, j)].set_facecolor('#F5F5F5')
                elif row_idx in device_rows:
                    table[(i, j)].set_facecolor('#F3E5F5')
                else:
                    table[(i, j)].set_facecolor('#FFFFFF')

        short_name = filename if len(filename) <= 50 else filename[:47] + "..."
        fig.suptitle(f'Maschinenparameter: {short_name}',
                    fontsize=13, fontweight='bold', y=0.99)

        plt.tight_layout()
        plt.show()

    if len(files_with_cnf) > 1:
        print(f"\n{'='*70}")
        print(f"📊 VERGLEICH Maschinenparameter")
        print(f"{'='*70}")

        compare_params = [
            ('Kraft @2 (upper) [N]',   lambda c: f"{c.kraft2_N:.1f}"),
            ('Kraft @1 (lower) [N]',   lambda c: f"{c.kraft1_N:.1f}"),
            ('Force ratio',            lambda c: f"1:{c.kraft1_ratio}"),
            ('Kraftabschaltung [N]',   lambda c: f"{c.kraftabschaltung_N:.1f}"),
            ('Toleranz [%]',           lambda c: f"± {c.toleranz_pct}"),
            ('NRW K1 [mm]',            lambda c: f"{c.nrw_k1_mm:.2f}"),
            ('NRW K2 [mm]',            lambda c: f"{c.nrw_k2_mm:.2f}"),
            ('Betätigungen',           lambda c: f"{c.betaetigungen:,}"),
            ('Messen alle N',          lambda c: f"{c.messen_alle_n}"),
            ('Device',                 lambda c: f"{c.device_version} {c.device_id}"),
        ]

        col_labels_cmp = ['Parameter'] + [
            (f if len(f) <= 20 else f[:17] + "...") for f in files_with_cnf
        ]

        table_data_cmp = []
        for label, getter in compare_params:
            row = [label]
            for fn in files_with_cnf:
                row.append(getter(cnf_dict[fn]))
            table_data_cmp.append(row)

        n_cols = len(col_labels_cmp)
        fig_width = max(10, 3 + 2.5 * len(files_with_cnf))
        fig_height = 1.5 + 0.35 * len(table_data_cmp)
        fig = plt.figure(figsize=(fig_width, fig_height))
        ax = fig.add_subplot(111)
        ax.axis('off')

        col_widths = [0.25] + [0.75 / max(1, len(files_with_cnf))] * len(files_with_cnf)

        table = ax.table(cellText=table_data_cmp, colLabels=col_labels_cmp,
                        cellLoc='center', loc='center',
                        colWidths=col_widths,
                        bbox=[0, 0, 1, 1])

        table.auto_set_font_size(False)
        table.set_fontsize(8)
        table.scale(1, 1.3)

        for j in range(n_cols):
            table[(0, j)].set_facecolor('#2E86AB')
            table[(0, j)].set_text_props(weight='bold', color='white')

        for i in range(1, len(table_data_cmp) + 1):
            for j in range(n_cols):
                table[(i, j)].set_facecolor('#F0F8FF' if i % 2 == 0 else '#FFFFFF')
            table[(i, 0)].set_text_props(ha='left')

        for i in range(1, len(table_data_cmp) + 1):
            values_in_row = [table_data_cmp[i-1][j] for j in range(1, n_cols)]
            if len(set(values_in_row)) > 1:
                for j in range(1, n_cols):
                    table[(i, j)].set_text_props(weight='bold', color='red')

        fig.suptitle('Vergleich Maschinenparameter',
                    fontsize=13, fontweight='bold', y=0.99)

        plt.tight_layout()
        plt.show()


# ============================================================================
# Dateiupload & Auswahl
# ============================================================================
def upload_and_parse():
    global data_dict, file_checkboxes, cnf_dict
    print("Bitte wähle eine oder mehrere Log-Dateien aus...")
    uploaded = files.upload()

    if uploaded:
        files_loaded = 0
        for filename in uploaded.keys():
            content = uploaded[filename].decode('utf-8')
            print(f"\nParse Datei: {filename}")
            df, cnf_config = parse_logfile(content)
            if df is not None and len(df) > 0:
                data_dict[filename] = df
                cnf_dict[filename] = cnf_config
                files_loaded += 1
                print(f"  ✓ {len(df)} Datenpunkte")
                if cnf_config.parse_ok:
                    print(f"  ✓ CNF-Header dekodiert: Kraft @2 = {cnf_config.kraft2_N:.1f} N, "
                          f"Kraft @1 = {cnf_config.kraft1_N:.1f} N, "
                          f"Betätigungen = {cnf_config.betaetigungen:,}")
                else:
                    print(f"  ⚠️ Kein CNF-Header gefunden")
        if files_loaded > 0:
            create_file_selection()
            enable_plot_controls()

def create_file_selection():
    global file_checkboxes
    file_checkboxes = {}
    for filename in data_dict.keys():
        checkbox = widgets.Checkbox(value=True, description=filename,
                                    style={'description_width': 'initial'},
                                    layout=widgets.Layout(width='100%'))
        file_checkboxes[filename] = checkbox
    update_file_selection_ui()

def update_file_selection_ui():
    global file_selection_container
    if len(file_checkboxes) > 0:
        file_selection_container.children = [
            widgets.HTML("<h4>📁 Geladene Dateien:</h4>"),
            widgets.VBox(list(file_checkboxes.values()))
        ]

def get_selected_files():
    return [filename for filename, checkbox in file_checkboxes.items() if checkbox.value]

def get_statistics_text():
    selected_files = get_selected_files()
    if not selected_files:
        return "Keine Dateien ausgewählt"
    stats_text = "WÖHLER-KURVEN DATENAUSWERTUNG\n" + "="*70 + "\n"
    for filename in selected_files:
        df = data_dict[filename]
        stats_text += f"\nDATEI: {filename}\nZyklen: {df['Zyklen'].max()}\nLoss: {df['Steifigkeitsverlust_Prozent'].iloc[-1]:.2f}%\n"

        if filename in cnf_dict and cnf_dict[filename].parse_ok:
            cfg = cnf_dict[filename]
            stats_text += f"Maschinenparameter:\n"
            stats_text += f"  Kraft @2 (upper):  {cfg.kraft2_N:.1f} N\n"
            stats_text += f"  Kraft @1 (lower):  {cfg.kraft1_N:.1f} N\n"
            stats_text += f"  Kraftabschaltung:  {cfg.kraftabschaltung_N:.1f} N\n"
            stats_text += f"  Toleranz:          ± {cfg.toleranz_pct} %\n"
            stats_text += f"  Betätigungen:      {cfg.betaetigungen:,}\n"
            stats_text += f"  Messen alle N:     {cfg.messen_alle_n}\n"
            stats_text += f"  Device:            {cfg.device_version} ID={cfg.device_id}\n"

            # DIN 50100 Prüfung
            din_eval = evaluate_din50100_criteria(df, cfg)
            if din_eval:
                stats_text += f"\nDIN 50100 Prüfung (Erlaubte Abweichung ±3% = ±{din_eval['allowed_dev']:.2f} N):\n"
                stats_text += f"  1. 3-Sigma ({din_eval['3sigma']:.2f} N) <= 3% Limit: {'✅ PASS' if din_eval['crit1_pass'] else '❌ FAIL'}\n"
                stats_text += f"  2. Max 10-Zyklen Ø Abweichung ({din_eval['max_roll_dev']:.2f} N) <= 3% Limit: {'✅ PASS' if din_eval['crit2_pass'] else '❌ FAIL'}\n"

    return stats_text


# ============================================================================
# DRUCK-OPTIMIERT: Kombinierte Plots
# ============================================================================
def create_combined_plots(show_pos1, show_kraft1, show_zusatz1,
                         show_pos2, show_kraft2, show_zusatz2, show_weg,
                         show_r_wert, show_kraftamplitude, show_mittelkraft,
                         show_steifigkeitsverlust, show_kriterium_10,
                         show_kriterium_20, show_kriterium_50, use_smoothing=False):
    selected_files = get_selected_files()
    if not selected_files:
        print("❌ Keine Dateien ausgewählt.")
        return

    active_plots = sum([show_pos1, show_kraft1, show_zusatz1, show_pos2, show_kraft2,
                        show_zusatz2, show_weg, show_r_wert, show_kraftamplitude,
                        show_mittelkraft, show_steifigkeitsverlust])

    if active_plots == 0:
        print("❌ Bitte wähle mindestens einen Graphen aus.")
        return

    print(f"🖨️ DRUCK-MODUS: Rasterisiert={RASTERIZE_PLOTS}, DPI={RASTER_DPI}, Transparenz={USE_TRANSPARENCY}")

    fig, axes = plt.subplots(active_plots, 1, figsize=(14, 10 * active_plots), dpi=SCREEN_DPI)
    if active_plots == 1:
        axes = [axes]

    plot_idx = 0
    alpha_val = 0.7 if USE_TRANSPARENCY else 1.0

    configs = [
        (show_pos1, 'Position_1_mm', 'Position 1 [mm]', 'Position 1'),
        (show_kraft1, 'Kraft_1_N', 'Kraft 1 [N]', 'Kraft 1 (F_min)'),
        (show_zusatz1, 'Zusatzweg_1_mm', 'Zusatzweg 1 [mm]', 'Zusatzweg 1'),
        (show_pos2, 'Position_2_mm', 'Position 2 [mm]', 'Position 2'),
        (show_kraft2, 'Kraft_2_N', 'Kraft 2 [N]', 'Kraft 2 (F_max)'),
        (show_zusatz2, 'Zusatzweg_2_mm', 'Zusatzweg 2 [mm]', 'Zusatzweg 2'),
        (show_weg, 'Weg_mm', 'Weg [mm]', 'Weg'),
        (show_r_wert, 'R_Wert', 'R-Wert', 'R-Wert'),
        (show_kraftamplitude, 'Kraftamplitude_N', 'F_a [N]', 'Kraftamplitude'),
        (show_mittelkraft, 'Mittelkraft_N', 'F_m [N]', 'Mittelkraft'),
        (show_steifigkeitsverlust, 'Steifigkeitsverlust_Prozent', 'Verlust [%]', 'Steifigkeitsverlust')
    ]

    for show, col, ylabel, title in configs:
        if show:
            for i, filename in enumerate(selected_files):
                df = data_dict[filename]
                if use_smoothing:
                    df = smooth_data(df, window_size=50)

                df_plot = downsample_data(df, MAX_PLOT_POINTS)

                axes[plot_idx].plot(df_plot['Zyklen'], df_plot[col],
                                          color=COLORS[i % len(COLORS)],
                                          label=filename, alpha=alpha_val,
                                          linewidth=0.8,
                                          rasterized=RASTERIZE_PLOTS)

            axes[plot_idx].set_xlabel('Zyklen', fontsize=11)
            axes[plot_idx].set_ylabel(ylabel, fontsize=11)
            axes[plot_idx].set_title(title, fontsize=13, fontweight='bold')
            axes[plot_idx].grid(True, alpha=0.3, linewidth=0.5)
            axes[plot_idx].legend(fontsize=8)

            create_statistics_table(axes[plot_idx], selected_files, col, data_dict, use_smoothing)
            plot_idx += 1

    plt.subplots_adjust(hspace=0.7)
    plt.show()

    print("\nZUSAMMENFASSUNG")
    create_summary_table(selected_files, data_dict, show_kriterium_10,
                        show_kriterium_20, show_kriterium_50, use_smoothing)


def create_all_qq_plots(show_qq_pos1, show_qq_kraft1, show_qq_zusatz1,
                       show_qq_pos2, show_qq_kraft2, show_qq_zusatz2,
                       show_qq_weg, show_qq_r_wert, show_qq_kraftamplitude,
                       show_qq_mittelkraft, use_smoothing=False):
    selected_files = get_selected_files()
    if not selected_files:
        print("❌ Keine Dateien ausgewählt.")
        return

    active_qq = sum([show_qq_pos1, show_qq_kraft1, show_qq_zusatz1,
                     show_qq_pos2, show_qq_kraft2, show_qq_zusatz2,
                     show_qq_weg, show_qq_r_wert, show_qq_kraftamplitude,
                     show_qq_mittelkraft])

    if active_qq == 0:
        print("❌ Bitte wähle mindestens einen Q-Q Plot aus.")
        return

    print(f"📊 Erstelle {active_qq} Q-Q Plot(s)...")
    print(f"🖨️ DRUCK-MODUS: Rasterisiert={RASTERIZE_PLOTS}, DPI={RASTER_DPI}")

    qq_configs = [
        (show_qq_pos1, 'Position_1_mm', 'Position 1 [mm]'),
        (show_qq_kraft1, 'Kraft_1_N', 'Kraft 1 (F_min) [N]'),
        (show_qq_zusatz1, 'Zusatzweg_1_mm', 'Zusatzweg 1 [mm]'),
        (show_qq_pos2, 'Position_2_mm', 'Position 2 [mm]'),
        (show_qq_kraft2, 'Kraft_2_N', 'Kraft 2 (F_max) [N]'),
        (show_qq_zusatz2, 'Zusatzweg_2_mm', 'Zusatzweg 2 [mm]'),
        (show_qq_weg, 'Weg_mm', 'Weg [mm]'),
        (show_qq_r_wert, 'R_Wert', 'R-Wert'),
        (show_qq_kraftamplitude, 'Kraftamplitude_N', 'Kraftamplitude (F_a) [N]'),
        (show_qq_mittelkraft, 'Mittelkraft_N', 'Mittelkraft (F_m) [N]'),
    ]

    for show, col, label in qq_configs:
        if show:
            create_qq_plots_for_parameter(selected_files, col, label, data_dict, use_smoothing)


# ============================================================================
# PDF Export
# ============================================================================
def save_plots_and_data():
    selected_files = get_selected_files()

    if not selected_files:
        print("❌ Keine Dateien ausgewählt.")
        return

    print("📄 Erstelle DRUCK-OPTIMIERTES PDF...")
    print(f"🖨️ Rasterisierung: {RASTERIZE_PLOTS} (DPI={RASTER_DPI})")

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    pdf_filename = f"Woehler_Vergleich_{timestamp}_print.pdf"

    show_pos1 = checkbox_pos1.value
    show_kraft1 = checkbox_kraft1.value
    show_zusatz1 = checkbox_zusatz1.value
    show_pos2 = checkbox_pos2.value
    show_kraft2 = checkbox_kraft2.value
    show_zusatz2 = checkbox_zusatz2.value
    show_weg = checkbox_weg.value
    show_r_wert = checkbox_r_wert.value
    show_kraftamplitude = checkbox_kraftamplitude.value
    show_mittelkraft = checkbox_mittelkraft.value
    show_steifigkeitsverlust = checkbox_steifigkeitsverlust.value
    show_kriterium_10 = checkbox_kriterium_10.value
    show_kriterium_20 = checkbox_kriterium_20.value
    show_kriterium_50 = checkbox_kriterium_50.value
    use_smoothing = checkbox_smooth.value

    show_qq_pos1 = checkbox_qq_pos1.value
    show_qq_kraft1 = checkbox_qq_kraft1.value
    show_qq_zusatz1 = checkbox_qq_zusatz1.value
    show_qq_pos2 = checkbox_qq_pos2.value
    show_qq_kraft2 = checkbox_qq_kraft2.value
    show_qq_zusatz2 = checkbox_qq_zusatz2.value
    show_qq_weg = checkbox_qq_weg.value
    show_qq_r_wert = checkbox_qq_r_wert.value
    show_qq_kraftamplitude = checkbox_qq_kraftamplitude.value
    show_qq_mittelkraft = checkbox_qq_mittelkraft.value

    show_drift_kraft1 = checkbox_drift_kraft1.value
    show_drift_kraft2 = checkbox_drift_kraft2.value
    show_drift_r_wert = checkbox_drift_r_wert.value
    show_drift_kraftamplitude = checkbox_drift_kraftamplitude.value

    is_comparison = len(selected_files) > 1
    alpha_val = 0.7 if USE_TRANSPARENCY else 1.0

    active_plots = sum([show_pos1, show_kraft1, show_zusatz1, show_pos2, show_kraft2,
                        show_zusatz2, show_weg, show_r_wert, show_kraftamplitude,
                        show_mittelkraft, show_steifigkeitsverlust])

    active_qq = sum([show_qq_pos1, show_qq_kraft1, show_qq_zusatz1,
                     show_qq_pos2, show_qq_kraft2, show_qq_zusatz2,
                     show_qq_weg, show_qq_r_wert, show_qq_kraftamplitude,
                     show_qq_mittelkraft])

    active_drift = sum([show_drift_kraft1, show_drift_kraft2,
                        show_drift_r_wert, show_drift_kraftamplitude])

    with PdfPages(pdf_filename) as pdf:
        files_with_cnf = [f for f in selected_files if f in cnf_dict and cnf_dict[f].parse_ok]
        machine_param_pages = 0

        if files_with_cnf:
            print(f"  ✓ Maschinenparameter ({len(files_with_cnf)} Datei(en))...")
            for filename in files_with_cnf:
                cfg = cnf_dict[filename]
                rows = cfg.to_display_rows()

                fig_height = 1.5 + 0.35 * len(rows)
                fig = plt.figure(figsize=(12, fig_height))
                ax = fig.add_subplot(111)
                ax.axis('off')

                table_data = [[label, value, unit] for label, value, unit in rows]
                col_labels = ['Parameter', 'Wert', 'Einheit / Bemerkung']

                table = ax.table(cellText=table_data, colLabels=col_labels,
                                cellLoc='left', loc='center',
                                colWidths=[0.35, 0.25, 0.40],
                                bbox=[0, 0, 1, 1])

                table.auto_set_font_size(False)
                table.set_fontsize(9)
                table.scale(1, 1.4)

                for j in range(3):
                    table[(0, j)].set_facecolor('#2E86AB')
                    table[(0, j)].set_text_props(weight='bold', color='white', fontsize=10)

                category_rows = {0, 1, 2, 3}
                safety_rows = {4, 5, 6}
                nachregelweg_rows = {7, 8}
                cycle_rows = {9, 10}
                device_rows = {11, 12}

                for i in range(1, len(table_data) + 1):
                    row_idx = i - 1
                    for j in range(3):
                        if row_idx in category_rows:
                            table[(i, j)].set_facecolor('#E3F2FD')
                        elif row_idx in safety_rows:
                            table[(i, j)].set_facecolor('#FFF9C4')
                        elif row_idx in nachregelweg_rows:
                            table[(i, j)].set_facecolor('#E8F5E9')
                        elif row_idx in cycle_rows:
                            table[(i, j)].set_facecolor('#F5F5F5')
                        elif row_idx in device_rows:
                            table[(i, j)].set_facecolor('#F3E5F5')
                        else:
                            table[(i, j)].set_facecolor('#FFFFFF')

                short_name = filename if len(filename) <= 50 else filename[:47] + "..."
                fig.suptitle(f'Maschinenparameter: {short_name}',
                            fontsize=13, fontweight='bold', y=0.99)

                plt.tight_layout()
                pdf.savefig(fig, bbox_inches='tight', dpi=RASTER_DPI)
                plt.close(fig)
                machine_param_pages += 1

        print("  ✓ Statistik-Seiten...")
        stats_text = get_statistics_text()
        lines = stats_text.split('\n')
        lines_per_page = 55

        for page_start in range(0, len(lines), lines_per_page):
            page_lines = lines[page_start:page_start + lines_per_page]
            fig = plt.figure(figsize=(8.5, 11), dpi=SCREEN_DPI)
            fig.text(0.1, 0.95, 'STATISTISCHE AUSWERTUNG - DRUCK-OPTIMIERT',
                    fontsize=15, fontweight='bold', va='top')

            total_text_pages = (len(lines) + lines_per_page - 1) // lines_per_page
            if total_text_pages > 1:
                page_num = (page_start // lines_per_page) + 1
                fig.text(0.1, 0.92, f'Seite {page_num} von {total_text_pages}',
                        fontsize=9, style='italic', va='top')

            stats_content = '\n'.join(page_lines)
            fig.text(0.1, 0.88, stats_content, fontsize=7, family='monospace', va='top')

            plt.axis('off')
            pdf.savefig(fig, bbox_inches='tight', dpi=SCREEN_DPI)
            plt.close(fig)

        stats_pages = (len(lines) + lines_per_page - 1) // lines_per_page

        if active_plots > 0:
            print(f"  ✓ Erstelle {active_plots} rasterisierte Plot(s)...")

            fig, axes = plt.subplots(active_plots, 1,
                                    figsize=(14, 10 * active_plots),
                                    dpi=SCREEN_DPI)
            if active_plots == 1:
                axes = [axes]

            plot_idx = 0

            configs = [
                (show_pos1, 'Position_1_mm', 'Position 1 [mm]', 'Position 1'),
                (show_kraft1, 'Kraft_1_N', 'Kraft 1 [N]', 'Kraft 1 (F_min)'),
                (show_zusatz1, 'Zusatzweg_1_mm', 'Zusatzweg 1 [mm]', 'Zusatzweg 1'),
                (show_pos2, 'Position_2_mm', 'Position 2 [mm]', 'Position 2'),
                (show_kraft2, 'Kraft_2_N', 'Kraft 2 [N]', 'Kraft 2 (F_max)'),
                (show_zusatz2, 'Zusatzweg_2_mm', 'Zusatzweg 2 [mm]', 'Zusatzweg 2'),
                (show_weg, 'Weg_mm', 'Weg [mm]', 'Weg'),
                (show_r_wert, 'R_Wert', 'R-Wert', 'R-Wert'),
                (show_kraftamplitude, 'Kraftamplitude_N', 'F_a [N]', 'Kraftamplitude'),
                (show_mittelkraft, 'Mittelkraft_N', 'F_m [N]', 'Mittelkraft'),
            ]

            for show, col, ylabel, title_base in configs:
                if show:
                    title_suffix = ' (geglättet)' if use_smoothing else ''
                    title = f'{title_base} über Zyklen'
                    if is_comparison:
                        title += ' - Vergleich'
                    title += title_suffix

                    for i, filename in enumerate(selected_files):
                        df = data_dict[filename]
                        if use_smoothing:
                            df = smooth_data(df, window_size=50)

                        df_plot = downsample_data(df, MAX_PLOT_POINTS)

                        axes[plot_idx].plot(df_plot['Zyklen'], df_plot[col],
                                          color=COLORS[i % len(COLORS)],
                                          label=filename, alpha=alpha_val,
                                          linewidth=0.8,
                                          rasterized=RASTERIZE_PLOTS)

                    axes[plot_idx].set_xlabel('Zyklen', fontsize=11)
                    axes[plot_idx].set_ylabel(ylabel, fontsize=11)
                    axes[plot_idx].set_title(title, fontsize=13, fontweight='bold')
                    axes[plot_idx].grid(True, alpha=0.3, linewidth=0.5)
                    axes[plot_idx].legend(fontsize=8)

                    create_statistics_table(axes[plot_idx], selected_files, col,
                                          data_dict, use_smoothing)
                    plot_idx += 1

            if show_steifigkeitsverlust:
                title_suffix = ' (geglättet)' if use_smoothing else ''
                title = 'Steifigkeitsverlust über Zyklen'
                if is_comparison:
                    title += ' - Vergleich'
                title += title_suffix

                for i, filename in enumerate(selected_files):
                    df = data_dict[filename]
                    if use_smoothing:
                        df = smooth_data(df, window_size=50)

                    df_plot = downsample_data(df, MAX_PLOT_POINTS)

                    axes[plot_idx].plot(df_plot['Zyklen'],
                                      df_plot['Steifigkeitsverlust_Prozent'],
                                      color=COLORS[i % len(COLORS)],
                                      label=filename, alpha=alpha_val,
                                      linewidth=1.2,
                                      rasterized=RASTERIZE_PLOTS)

                if show_kriterium_10:
                    axes[plot_idx].axhline(y=10, color='orange', linestyle=':',
                                          linewidth=1, alpha=0.5, label='10% Kriterium')
                if show_kriterium_20:
                    axes[plot_idx].axhline(y=20, color='red', linestyle=':',
                                          linewidth=1, alpha=0.5, label='20% Kriterium')
                if show_kriterium_50:
                    axes[plot_idx].axhline(y=50, color='darkred', linestyle=':',
                                          linewidth=1, alpha=0.5, label='50% Kriterium')

                axes[plot_idx].set_xlabel('Zyklen', fontsize=11)
                axes[plot_idx].set_ylabel('Steifigkeitsverlust [%]', fontsize=11)
                axes[plot_idx].set_title(title, fontsize=13, fontweight='bold')
                axes[plot_idx].grid(True, alpha=0.3, linewidth=0.5)
                axes[plot_idx].legend(loc='best', fontsize=8)

                create_statistics_table(axes[plot_idx], selected_files,
                                      'Steifigkeitsverlust_Prozent',
                                      data_dict, use_smoothing)

            plt.subplots_adjust(hspace=0.7)
            pdf.savefig(fig, bbox_inches='tight', dpi=RASTER_DPI)
            plt.close(fig)

        if active_qq > 0:
            print(f"  ✓ Erstelle {active_qq} Q-Q Plot(s)...")

            qq_configs = [
                (show_qq_pos1, 'Position_1_mm', 'Position 1 [mm]'),
                (show_qq_kraft1, 'Kraft_1_N', 'Kraft 1 (F_min) [N]'),
                (show_qq_zusatz1, 'Zusatzweg_1_mm', 'Zusatzweg 1 [mm]'),
                (show_qq_pos2, 'Position_2_mm', 'Position 2 [mm]'),
                (show_qq_kraft2, 'Kraft_2_N', 'Kraft 2 (F_max) [N]'),
                (show_qq_zusatz2, 'Zusatzweg_2_mm', 'Zusatzweg 2 [mm]'),
                (show_qq_weg, 'Weg_mm', 'Weg [mm]'),
                (show_qq_r_wert, 'R_Wert', 'R-Wert'),
                (show_qq_kraftamplitude, 'Kraftamplitude_N', 'Kraftamplitude (F_a) [N]'),
                (show_qq_mittelkraft, 'Mittelkraft_N', 'Mittelkraft (F_m) [N]'),
            ]

            for show, col, label in qq_configs:
                if show:
                    n_files = len(selected_files)
                    fig = plt.figure(figsize=(14, 5 * n_files), dpi=SCREEN_DPI)

                    for idx, filename in enumerate(selected_files):
                        df = data_dict[filename]
                        if use_smoothing:
                            df = smooth_data(df, window_size=50)

                        gs = GridSpec(n_files, 2, figure=fig,
                                     width_ratios=[2, 1],
                                     hspace=0.4, wspace=0.3,
                                     left=0.1, right=0.95, top=0.95, bottom=0.05)

                        ax_qq = fig.add_subplot(gs[idx, 0])
                        ax_hist = fig.add_subplot(gs[idx, 1])

                        color = COLORS[idx % len(COLORS)]
                        smooth_text = ' (geglättet)' if use_smoothing else ''
                        title = f'Q-Q: {label}{smooth_text} - {filename}'

                        create_qq_plot_with_histogram(ax_qq, ax_hist, df[col], title, color)

                    pdf.savefig(fig, bbox_inches='tight', dpi=RASTER_DPI)
                    plt.close(fig)

        if active_drift > 0:
            print(f"  ✓ Erstelle {active_drift} Drift-Analyse(n)...")

            drift_configs = [
                (show_drift_kraft1, 'Kraft_1_N', 'Kraft 1 [N]'),
                (show_drift_kraft2, 'Kraft_2_N', 'Kraft 2 [N]'),
                (show_drift_r_wert, 'R_Wert', 'R-Wert'),
                (show_drift_kraftamplitude, 'Kraftamplitude_N', 'Kraftamplitude [N]'),
            ]

            for show, col_name, col_label in drift_configs:
                if show:
                    n_files = len(selected_files)
                    fig, axes = plt.subplots(n_files, 1,
                                            figsize=(14, 6 * n_files),
                                            dpi=SCREEN_DPI)
                    if n_files == 1:
                        axes = [axes]

                    drift_data_list = []

                    for idx, filename in enumerate(selected_files):
                        df = data_dict[filename]
                        if use_smoothing:
                            df = smooth_data(df, window_size=50)

                        color = COLORS[idx % len(COLORS)]
                        drift = create_drift_plot(axes[idx], df, col_name,
                                                 f"{col_label} - {filename}", color)

                        if drift:
                            drift_data_list.append((filename, drift))

                    smooth_text = ' (geglättet)' if use_smoothing else ''
                    fig.suptitle(f'Drift-Analyse: {col_label}{smooth_text}',
                                fontsize=16, fontweight='bold', y=0.995)

                    plt.tight_layout()
                    pdf.savefig(fig, bbox_inches='tight', dpi=RASTER_DPI)
                    plt.close(fig)

                    if drift_data_list:
                        fig_height = 2 + 0.35 * len(drift_data_list)
                        fig = plt.figure(figsize=(16, fig_height), dpi=SCREEN_DPI)
                        ax = fig.add_subplot(111)
                        ax.axis('off')

                        table_data = []
                        headers = ['Datei', 'Start', 'Ende', 'Drift abs.', 'Drift %',
                                   'pro 1000 Zyklen', 'Trend', 'R²', 'Signifikanz']

                        for filename, drift in drift_data_list:
                            short_name = filename if len(filename) <= 25 else filename[:22] + "..."
                            row = [
                                short_name,
                                f"{drift['start_value']:.2f}",
                                f"{drift['end_value']:.2f}",
                                f"{drift['total_drift']:+.3f}",
                                f"{drift['relative_drift_percent']:+.2f}%",
                                f"{drift['drift_per_1000']:+.4f}",
                                drift['trend'],
                                f"{drift['r_squared']:.3f}",
                                drift['significance']
                            ]
                            table_data.append(row)

                        table = ax.table(cellText=table_data, colLabels=headers,
                                       cellLoc='center', loc='center',
                                       bbox=[0, 0, 1, 1])

                        table.auto_set_font_size(False)
                        table.set_fontsize(7)
                        table.scale(1, 1.2)

                        for i in range(len(headers)):
                            table[(0, i)].set_facecolor('#FF6B6B')
                            table[(0, i)].set_text_props(weight='bold', color='white')

                        for i in range(1, len(table_data) + 1):
                            for j in range(len(headers)):
                                if i % 2 == 0:
                                    table[(i, j)].set_facecolor('#FFE5E5')
                                else:
                                    table[(i, j)].set_facecolor('#FFFFFF')

                        fig.suptitle(f'Drift-Statistik: {col_label}{smooth_text}',
                                    fontsize=14, fontweight='bold', y=0.98)

                        plt.tight_layout()
                        pdf.savefig(fig, bbox_inches='tight', dpi=RASTER_DPI)
                        plt.close(fig)

        print(f"  ✓ Zusammenfassungs-Tabelle...")

        fig_height = 2 + 0.4 * len(selected_files)
        fig = plt.figure(figsize=(16, fig_height), dpi=SCREEN_DPI)
        ax = fig.add_subplot(111)
        ax.axis('off')

        table_data = []
        headers = ['Datei', 'Kraft 1 Ø [N]', 'Kraft 1 σ [N]',
                   'Kraft 2 Ø [N]', 'Kraft 2 σ [N]',
                   'DIN 3σ Check', 'DIN 10-Zykl. Check',
                   'F_a Ø [N]', 'F_a σ [N]']

        if show_kriterium_10: headers.append('Zyklen @ 10%')
        if show_kriterium_20: headers.append('Zyklen @ 20%')
        if show_kriterium_50: headers.append('Zyklen @ 50%')

        for filename in selected_files:
            df = data_dict[filename]
            if use_smoothing:
                df = smooth_data(df, window_size=50)

            short_name = filename if len(filename) <= 25 else filename[:22] + "..."

            din_3sigma_str = "---"
            din_10cyc_str = "---"
            if filename in cnf_dict and cnf_dict[filename].parse_ok:
                din_eval = evaluate_din50100_criteria(df, cnf_dict[filename])
                if din_eval:
                    din_3sigma_str = "PASS" if din_eval['crit1_pass'] else "FAIL"
                    din_10cyc_str = "PASS" if din_eval['crit2_pass'] else "FAIL"

            row = [short_name,
                   f"{df['Kraft_1_N'].mean():.2f}", f"{df['Kraft_1_N'].std():.2f}",
                   f"{df['Kraft_2_N'].mean():.2f}", f"{df['Kraft_2_N'].std():.2f}",
                   din_3sigma_str, din_10cyc_str,
                   f"{df['Kraftamplitude_N'].mean():.2f}", f"{df['Kraftamplitude_N'].std():.2f}"]

            for thresh, show in [(10, show_kriterium_10), (20, show_kriterium_20), (50, show_kriterium_50)]:
                if show:
                    idx = df[df['Steifigkeitsverlust_Prozent'] >= thresh].first_valid_index()
                    row.append(f"{int(df.loc[idx, 'Zyklen']):,}" if idx is not None else "---")

            table_data.append(row)

        table = ax.table(cellText=table_data, colLabels=headers, cellLoc='center',
                         loc='center', bbox=[0, 0, 1, 1])
        table.auto_set_font_size(False)
        table.set_fontsize(7)
        table.scale(1, 1.2)

        for i in range(len(headers)):
            table[(0, i)].set_facecolor('#4472C4')
            table[(0, i)].set_text_props(weight='bold', color='white')

        for i in range(1, len(table_data) + 1):
            for j in range(len(headers)):
                table[(i, j)].set_facecolor('#E7E6E6' if i % 2 == 0 else '#FFFFFF')

                cell_text = table_data[i-1][j]
                if cell_text == "PASS":
                    table[(i, j)].set_text_props(color='green', weight='bold')
                elif cell_text == "FAIL":
                    table[(i, j)].set_text_props(color='red', weight='bold')

        title = 'Zusammenfassung'
        if is_comparison:
            title += ' - Vergleich'
        if use_smoothing:
            title += ' (geglättet)'
        fig.suptitle(title, fontsize=13, fontweight='bold', y=0.98)

        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight', dpi=RASTER_DPI)
        plt.close(fig)

        d = pdf.infodict()
        d['Title'] = 'Wöhler-Kurven - DRUCK-OPTIMIERT mit Drift-Analyse, Maschinenparameter & DIN 50100'
        d['Author'] = 'Voice Coil Test Stand Analyzer'
        d['Subject'] = f'Druck-optimiert: Rasterisiert (DPI={RASTER_DPI})'
        d['Keywords'] = 'Wöhler, Print-Optimized, Rasterized, Drift, Machine Config, DIN50100'
        d['CreationDate'] = datetime.now()

    drift_pages = active_drift * 2 if active_drift > 0 else 0
    total_pages = (machine_param_pages + stats_pages +
                   (1 if active_plots > 0 else 0) + active_qq + drift_pages + 1)

    print(f"✅ DRUCK-OPTIMIERTES PDF: {pdf_filename}")
    print(f"   📊 {total_pages} Seiten gesamt:")
    if machine_param_pages > 0:
        print(f"      • {machine_param_pages} Maschinenparameter")
    print(f"      • {stats_pages} Statistik")
    if active_plots > 0:
        print(f"      • 1 Standard-Plots")
    if active_qq > 0:
        print(f"      • {active_qq} Q-Q Plots")
    if active_drift > 0:
        print(f"      • {drift_pages} Drift-Analyse ({active_drift} Parameter)")
    print(f"      • 1 Zusammenfassung")
    print(f"   🖨️ Rasterisiert mit {RASTER_DPI} DPI")
    print(f"\n⬇️ Download...")

    files.download(pdf_filename)
    print(f"✅ Fertig!")


# ============================================================================
# UI Definition
# ============================================================================
upload_button = widgets.Button(description='📁 Dateien auswählen',
                               button_style='success', icon='upload',
                               layout=widgets.Layout(width='450px', height='80px'))
upload_output = widgets.Output()
file_selection_container = widgets.VBox([widgets.HTML("<p><i>Noch keine Dateien geladen</i></p>")])

checkbox_pos1 = widgets.Checkbox(value=False, description='Position 1 [mm]', disabled=True)
checkbox_kraft1 = widgets.Checkbox(value=True, description='Kraft 1 (F_min) [N]', disabled=True)
checkbox_zusatz1 = widgets.Checkbox(value=False, description='Zusatzweg 1 [mm]', disabled=True)
checkbox_pos2 = widgets.Checkbox(value=False, description='Position 2 [mm]', disabled=True)
checkbox_kraft2 = widgets.Checkbox(value=True, description='Kraft 2 (F_max) [N]', disabled=True)
checkbox_zusatz2 = widgets.Checkbox(value=False, description='Zusatzweg 2 [mm]', disabled=True)
checkbox_weg = widgets.Checkbox(value=True, description='Weg [mm]', disabled=True)
checkbox_r_wert = widgets.Checkbox(value=True, description='R-Wert (σ_min/σ_max)', disabled=True)
checkbox_kraftamplitude = widgets.Checkbox(value=False, description='Kraftamplitude F_a', disabled=True)
checkbox_mittelkraft = widgets.Checkbox(value=False, description='Mittelkraft F_m', disabled=True)
checkbox_steifigkeitsverlust = widgets.Checkbox(value=True, description='Steifigkeitsverlust [%]', disabled=True)
checkbox_kriterium_10 = widgets.Checkbox(value=True, description='10% Steifigkeitsverlust', disabled=True)
checkbox_kriterium_20 = widgets.Checkbox(value=True, description='20% Steifigkeitsverlust', disabled=True)
checkbox_kriterium_50 = widgets.Checkbox(value=False, description='50% Steifigkeitsverlust', disabled=True)
checkbox_smooth = widgets.Checkbox(value=False, description='Daten glätten',
                                  disabled=True, style={'description_width': 'initial'})

checkbox_qq_pos1 = widgets.Checkbox(value=False, description='Q-Q: Position 1', disabled=True)
checkbox_qq_kraft1 = widgets.Checkbox(value=False, description='Q-Q: Kraft 1', disabled=True)
checkbox_qq_zusatz1 = widgets.Checkbox(value=False, description='Q-Q: Zusatzweg 1', disabled=True)
checkbox_qq_pos2 = widgets.Checkbox(value=False, description='Q-Q: Position 2', disabled=True)
checkbox_qq_kraft2 = widgets.Checkbox(value=False, description='Q-Q: Kraft 2', disabled=True)
checkbox_qq_zusatz2 = widgets.Checkbox(value=False, description='Q-Q: Zusatzweg 2', disabled=True)
checkbox_qq_weg = widgets.Checkbox(value=False, description='Q-Q: Weg', disabled=True)
checkbox_qq_r_wert = widgets.Checkbox(value=False, description='Q-Q: R-Wert', disabled=True)
checkbox_qq_kraftamplitude = widgets.Checkbox(value=False, description='Q-Q: Kraftamplitude', disabled=True)
checkbox_qq_mittelkraft = widgets.Checkbox(value=False, description='Q-Q: Mittelkraft', disabled=True)

checkbox_drift_kraft1 = widgets.Checkbox(value=True, description='Drift: Kraft 1 (F_min)', disabled=True)
checkbox_drift_kraft2 = widgets.Checkbox(value=True, description='Drift: Kraft 2 (F_max)', disabled=True)
checkbox_drift_r_wert = widgets.Checkbox(value=True, description='Drift: R-Wert', disabled=True)
checkbox_drift_kraftamplitude = widgets.Checkbox(value=True, description='Drift: Kraftamplitude', disabled=True)

plot_button = widgets.Button(description='📊 Plots erstellen', button_style='primary',
                             icon='chart-line', layout=widgets.Layout(width='220px', height='40px'),
                             disabled=True)
qq_button = widgets.Button(description='📉 Q-Q Plots', button_style='info',
                          icon='chart-area', layout=widgets.Layout(width='220px', height='40px'),
                          disabled=True)
drift_button = widgets.Button(description='📈 Drift-Analyse', button_style='warning',
                              icon='trending-up', layout=widgets.Layout(width='220px', height='40px'),
                              disabled=True)

machine_params_button = widgets.Button(
    description='🔧 Maschinenparameter',
    button_style='info',
    icon='cogs',
    layout=widgets.Layout(width='220px', height='40px'),
    disabled=True
)

save_button = widgets.Button(description='🖨️ PDF (Druck-optimiert)', button_style='success',
                            icon='print', layout=widgets.Layout(width='220px', height='40px'),
                            disabled=True)
plot_output = widgets.Output()

def on_upload_button_clicked(b):
    with upload_output: clear_output(); upload_and_parse()

def on_plot_button_clicked(b):
    with plot_output:
        clear_output(wait=True)
        create_combined_plots(checkbox_pos1.value, checkbox_kraft1.value, checkbox_zusatz1.value,
                            checkbox_pos2.value, checkbox_kraft2.value, checkbox_zusatz2.value,
                            checkbox_weg.value, checkbox_r_wert.value, checkbox_kraftamplitude.value,
                            checkbox_mittelkraft.value, checkbox_steifigkeitsverlust.value,
                            checkbox_kriterium_10.value, checkbox_kriterium_20.value,
                            checkbox_kriterium_50.value, checkbox_smooth.value)

def on_qq_button_clicked(b):
    with plot_output:
        clear_output(wait=True)
        create_all_qq_plots(checkbox_qq_pos1.value, checkbox_qq_kraft1.value, checkbox_qq_zusatz1.value,
                          checkbox_qq_pos2.value, checkbox_qq_kraft2.value, checkbox_qq_zusatz2.value,
                          checkbox_qq_weg.value, checkbox_qq_r_wert.value, checkbox_qq_kraftamplitude.value,
                          checkbox_qq_mittelkraft.value, checkbox_smooth.value)

def on_drift_button_clicked(b):
    with plot_output:
        clear_output(wait=True)
        create_all_drift_plots(checkbox_drift_kraft1.value, checkbox_drift_kraft2.value,
                              checkbox_drift_r_wert.value, checkbox_drift_kraftamplitude.value,
                              checkbox_smooth.value)

def on_machine_params_button_clicked(b):
    with plot_output:
        clear_output(wait=True)
        show_machine_parameters()

def on_save_button_clicked(b):
    save_plots_and_data()

def enable_plot_controls():
    for cb in [checkbox_pos1, checkbox_kraft1, checkbox_zusatz1, checkbox_pos2, checkbox_kraft2,
               checkbox_zusatz2, checkbox_weg, checkbox_r_wert, checkbox_kraftamplitude,
               checkbox_mittelkraft, checkbox_steifigkeitsverlust, checkbox_kriterium_10,
               checkbox_kriterium_20, checkbox_kriterium_50, checkbox_smooth,
               checkbox_qq_pos1, checkbox_qq_kraft1, checkbox_qq_zusatz1, checkbox_qq_pos2,
               checkbox_qq_kraft2, checkbox_qq_zusatz2, checkbox_qq_weg, checkbox_qq_r_wert,
               checkbox_qq_kraftamplitude, checkbox_qq_mittelkraft,
               checkbox_drift_kraft1, checkbox_drift_kraft2, checkbox_drift_r_wert,
               checkbox_drift_kraftamplitude]:
        cb.disabled = False
    plot_button.disabled = False
    qq_button.disabled = False
    drift_button.disabled = False
    machine_params_button.disabled = False
    save_button.disabled = False

upload_button.on_click(on_upload_button_clicked)
plot_button.on_click(on_plot_button_clicked)
qq_button.on_click(on_qq_button_clicked)
drift_button.on_click(on_drift_button_clicked)
machine_params_button.on_click(on_machine_params_button_clicked)
save_button.on_click(on_save_button_clicked)


# ============================================================================
# UI
# ============================================================================
def create_ui():
    header = widgets.HTML(f"""
    <h2>🔬 Wöhler-Kurven Datenauswertung - DRUCK-OPTIMIERT</h2>
    <p><b>🖨️ SCHNELLES DRUCKEN aus Acrobat!</b></p>
    <p style='background-color:#ccffcc; padding:10px; border-radius:5px;'>
    <b>✅ LÖSUNG FÜR LANGSAMES DRUCKEN:</b><br>
    • Plots werden als <b>Rastergrafiken</b> (nicht Vektoren) gespeichert<br>
    • <b>Rasterisierung</b>: {RASTERIZE_PLOTS} @ {RASTER_DPI} DPI<br>
    • <b>Keine Transparenzen</b> (schnelleres Rendering)<br>
    • → <b>Drucken aus Acrobat ist jetzt 10-20x schneller!</b> ⚡
    </p>
    <p style='background-color:#e3f2fd; padding:10px; border-radius:5px;'>
    <b>🔧 NEU: Maschinenparameter & DIN 50100</b><br>
    • Dekodiert CNF-Header aus Log-Dateien automatisch<br>
    • Zeigt Kraft-Sollwerte, Toleranzen, Sicherheitsgrenzwerte, Zyklen-Einstellungen<br>
    • Führt automatisch eine ±3% Konformitätsprüfung für F_max aus (3-Sigma und 10-Zyklen Ø)<br>
    • Vergleichstabelle bei mehreren Dateien (Unterschiede rot markiert)
    </p>
    <hr>
    """)

    messdaten_header = widgets.HTML("<h4>📊 Messdaten</h4>")
    messdaten_box = widgets.HBox([
        widgets.VBox([checkbox_pos1, checkbox_kraft1, checkbox_zusatz1]),
        widgets.VBox([checkbox_pos2, checkbox_kraft2, checkbox_zusatz2]),
        widgets.VBox([checkbox_weg])
    ])

    woehler_header = widgets.HTML("<h4>📈 Wöhler-Parameter</h4>")
    woehler_box = widgets.HBox([checkbox_r_wert, checkbox_kraftamplitude, checkbox_mittelkraft])

    steifigkeit_header = widgets.HTML("<h4>🔧 Steifigkeit</h4>")
    steifigkeit_box = widgets.VBox([
        checkbox_steifigkeitsverlust,
        widgets.HBox([checkbox_kriterium_10, checkbox_kriterium_20, checkbox_kriterium_50])
    ])

    qq_header = widgets.HTML("<h4>📉 Q-Q Plots</h4>")
    qq_messdaten_box = widgets.HBox([
        widgets.VBox([checkbox_qq_pos1, checkbox_qq_kraft1, checkbox_qq_zusatz1]),
        widgets.VBox([checkbox_qq_pos2, checkbox_qq_kraft2, checkbox_qq_zusatz2]),
        widgets.VBox([checkbox_qq_weg])
    ])
    qq_woehler_box = widgets.HBox([checkbox_qq_r_wert, checkbox_qq_kraftamplitude, checkbox_qq_mittelkraft])

    drift_header = widgets.HTML("""
        <h4>📈 Drift-Analyse (Trend über Zeit)</h4>
        <p><i>Analysiert systematische Änderungen der Werte über die Zyklen</i></p>
    """)
    drift_info = widgets.HTML("""
        <div style='background-color:#fff3cd; padding:10px; border-radius:5px; margin-bottom:10px;'>
        <b>Was wird berechnet:</b><br>
        • <b>Lineare Regression</b>: Trend-Linie y = a·x + b<br>
        • <b>Drift-Rate</b>: Absolute Änderung pro 1000 Zyklen<br>
        • <b>Relative Drift</b>: Prozentuale Änderung vom Startwert<br>
        • <b>Signifikanz</b>: R² und statistische Tests
        </div>
    """)
    drift_box = widgets.HBox([
        checkbox_drift_kraft1,
        checkbox_drift_kraft2,
        checkbox_drift_r_wert,
        checkbox_drift_kraftamplitude
    ])

    datenverarbeitung_header = widgets.HTML("<h4>⚙️ Datenverarbeitung</h4>")
    datenverarbeitung_box = widgets.VBox([checkbox_smooth])

    button_row = widgets.HBox([
        plot_button, qq_button, drift_button,
        machine_params_button,
        save_button
    ])

    ui = widgets.VBox([
        header,
        widgets.HTML("<h3>1️⃣ Dateien laden</h3>"),
        upload_button, upload_output,
        widgets.HTML("<br><h3>2️⃣ Dateien auswählen</h3>"),
        file_selection_container,
        widgets.HTML("<br><h3>3️⃣ Graphen auswählen</h3>"),
        messdaten_header, messdaten_box,
        woehler_header, woehler_box,
        steifigkeit_header, steifigkeit_box,
        widgets.HTML("<br>"),
        qq_header, qq_messdaten_box, qq_woehler_box,
        widgets.HTML("<br>"),
        drift_header, drift_info, drift_box,
        widgets.HTML("<br>"),
        datenverarbeitung_header, datenverarbeitung_box,
        widgets.HTML("<br><h3>4️⃣ Plots erstellen</h3>"),
        button_row,
        widgets.HTML("<br><h3>5️⃣ Visualisierung</h3>"),
        plot_output
    ])
    display(ui)

create_ui()